# C.3 follow-up v3 — per-seed fan-out + Path β at multiple lr_pulls

**Why v3.** v2 ran 3 conditions × 1 process each (10 seeds serialized inside each process), only used ~2.2 GB of the A100's 80 GB. v3 fans each n=10 condition out to **10 subprocesses × 1 seed each**, and uses the freed budget to test **4 conditions** instead of 3 (added a wikitext mid-strength cell and bumped wikitext_base from n=3 to n=10).

**4 conditions × 10 seeds = 40 parallel subprocesses, all on CUDA:**

| tag | corpus | lr_pull | n_events | n_seeds | hypothesis being tested |
|---|---|---:|---:|---:|---|
| `n10_synthetic_cellD` | synthetic | 1.0 | 3000 | 10 | confirms Cell D at graduation n |
| `n10_wikitext_best` | wikitext | 1.0 | 3000 | 10 | **Path β headline** at best-of-sweep |
| `n10_wikitext_base` | wikitext | 0.1 | 1000 | 10 | does wikitext alone break null at default lr_pull? |
| `n10_wikitext_mid` | wikitext | 0.5 | 1000 | 10 | maps lr_pull strength dependence on wikitext |

**Reliability machinery:**
- Cell 1 applies all 3 patches (CLI flags + kernel-trick eigvalsh + Salesforce/wikitext namespace).
- Cell 4 pre-warms the wikitext-2 HF cache (~/.cache/huggingface/) so the 30 subprocesses that need it share the on-disk cache instead of each re-downloading.
- Cell 7 launches all 40 procs with 1.5 s stagger (avoids CUDA-init races), polls every 30 s, auto-prints log tail on any failure.
- Cell 9 aggregates per-seed JSONs into pooled Wilson-CI tables — the merged result is the n=10 headline.

**Graduation gate.** Per [phase-3-deep-dive.md:180-189](https://github.com/Dypatterson/Neuro-AI/blob/main/notes/emergent-codebook/phase-3-deep-dive.md), CI-disjoint in any stratum where both std and ctrl have populated trials = Phase 3 graduation candidate.

In [ ]:
# 1. Clone the repo and apply three patches:
#    (a) --lr-pull / --lr-push CLI flags in the C.3 driver.
#    (b) Kernel-trick eigvalsh fix in consolidation.py (CUDA-stable).
#    (c) Switch wikitext loader to "Salesforce/wikitext" namespace.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git checkout codex/phase5-prime-bundle-first-scene-memory
!git log --oneline -3

import subprocess
pre_pull = subprocess.run(['grep', '-c', '--', '--lr-pull', 'experiments/c3_phase3_exit_criterion.py'], capture_output=True, text=True).stdout.strip()
pre_gram = subprocess.run(['grep', '-c', 'kernel trick', 'src/energy_memory/phase4/consolidation.py'], capture_output=True, text=True).stdout.strip()
pre_wt   = subprocess.run(['grep', '-c', 'Salesforce/wikitext', 'src/energy_memory/phase2/corpus.py'], capture_output=True, text=True).stdout.strip()
print(f'pre-patch tracers: --lr-pull = {pre_pull}; kernel-trick gram = {pre_gram}; Salesforce/wikitext = {pre_wt}')

patch = r'''diff --git a/experiments/c3_phase3_exit_criterion.py b/experiments/c3_phase3_exit_criterion.py
--- a/experiments/c3_phase3_exit_criterion.py
+++ b/experiments/c3_phase3_exit_criterion.py
@@ -576,6 +576,8 @@ def _run_single_seed_condition(
     k: int,
     alpha_anti: float,
     repulsion_step_size: float,
+    lr_pull: float,
+    lr_push: float,
     device: str,
     repo_root: Path,
     wikitext_corpus: Optional[_WikiTextCorpus] = None,
@@ -756,6 +758,8 @@ def _run_single_seed_condition(
             vocab_size=vocab_size,
             n_events=n_consolidation_events,
             device=device,
+            lr_pull=lr_pull,
+            lr_push=lr_push,
             repulsion_step_size=repulsion_step_size,
         )
 
@@ -838,6 +842,8 @@ def run(
     n_consolidation_events: int = 1000,
     alpha_anti: float = 0.0,
     repulsion_step_size: float = 0.0,
+    lr_pull: float = 0.1,
+    lr_push: float = 0.05,
     device: str,
     output_dir: Path,
     repo_root: Path,
@@ -919,6 +925,8 @@ def run(
                     k=k,
                     alpha_anti=alpha_anti,
                     repulsion_step_size=repulsion_step_size,
+                    lr_pull=lr_pull,
+                    lr_push=lr_push,
                     device=device,
                     repo_root=repo_root,
                     wikitext_corpus=wikitext_corpus,
@@ -1010,6 +1018,8 @@ def run(
             "substrate_repulsion_active": bool(
                 alpha_anti > 0.0 and repulsion_step_size > 0.0
             ),
+            "lr_pull": float(lr_pull),
+            "lr_push": float(lr_push),
             "operating_point": {
                 "D": D,
                 "landscape_size": landscape_size,
@@ -1370,6 +1380,27 @@ def main(argv: Optional[Sequence[str]] = None) -> int:
             "smoke (no inter-atom-separability force)."
         ),
     )
+    parser.add_argument(
+        "--lr-pull",
+        type=float,
+        default=0.1,
+        help=(
+            "Per-event consolidation pull learning rate (OnlineCodebookUpdater "
+            "lr_pull). Default 0.1 matches the existing Path α smoke. Sweep "
+            "above this to test whether consolidation strength is too weak "
+            "to express corpus-specific learning at the synthetic operating "
+            "point."
+        ),
+    )
+    parser.add_argument(
+        "--lr-push",
+        type=float,
+        default=0.05,
+        help=(
+            "Per-event consolidation push learning rate (OnlineCodebookUpdater "
+            "lr_push). Default 0.05 matches the existing Path α smoke."
+        ),
+    )
     parser.add_argument(
         "--repulsion-step-size",
         type=float,
@@ -1468,6 +1499,8 @@ def main(argv: Optional[Sequence[str]] = None) -> int:
         n_consolidation_events=args.n_consolidation_events,
         alpha_anti=args.alpha_anti,
         repulsion_step_size=args.repulsion_step_size,
+        lr_pull=args.lr_pull,
+        lr_push=args.lr_push,
         device=args.device,
         output_dir=output_dir,
         repo_root=repo_root,
diff --git a/src/energy_memory/phase4/consolidation.py b/src/energy_memory/phase4/consolidation.py
--- a/src/energy_memory/phase4/consolidation.py
+++ b/src/energy_memory/phase4/consolidation.py
@@ -639,10 +639,29 @@ class ConsolidationState:
         # Hermitian Gram of centered basin members. For complex (FHRR)
         # tensors, diffs.conj().T @ diffs is Hermitian → real eigenvalues
         # via torch.linalg.eigh.
-        sigma = (diffs.conj().transpose(-1, -2) @ diffs) / float(n)
+        # Compute the eigenvalues of σ = diffs.conj().T @ diffs / n via the
+        # n×n Gram matrix gram = diffs @ diffs.conj().T / n instead of the
+        # D×D scatter matrix. The two matrices share exactly the same set
+        # of non-zero eigenvalues (standard "kernel trick" identity); the
+        # D×D form additionally carries (D - n) trivial zero eigenvalues
+        # because rank(σ) ≤ n_members ≤ basin_trace_buffer_size (64) ≪ D
+        # (4096 by default in this project). That (D - n) zero subspace
+        # makes σ numerically ill-conditioned at the precision available
+        # to torch.linalg.eigvalsh — observed on Colab CUDA at 2026-05-27
+        # as LinAlgError 4095 and even on CPU LAPACK as LinAlgError 5/12.
+        # The n×n Gram path is full-rank for non-degenerate samples and
+        # an order of magnitude smaller (4 KB vs 16 MB at D=4096, n=8).
+        # Mathematically byte-identical at the λ_1 / λ_2 layer used below;
+        # the C.2.2 dynamic's behavior is unchanged.
+        gram = (diffs @ diffs.conj().transpose(-1, -2)) / float(n)
         # Eigh returns ascending eigenvalues. Take top two: λ_1 (last),
         # λ_2 (second-to-last). All ops stay on-device.
-        eigvals = torch.linalg.eigvalsh(sigma)
+        try:
+            eigvals = torch.linalg.eigvalsh(gram)
+        except torch._C._LinAlgError:
+            # Defensive: keep the CPU fallback in case some pathological
+            # input still trips cuSOLVER (e.g. identical basin members).
+            eigvals = torch.linalg.eigvalsh(gram.cpu()).to(gram.device)
         lam_1 = eigvals[-1]
         lam_2 = eigvals[-2] if eigvals.shape[0] >= 2 else torch.zeros_like(lam_1)
         # Clamp at 0 — eigh may return tiny negatives for near-singular Σ.
diff --git a/src/energy_memory/phase2/corpus.py b/src/energy_memory/phase2/corpus.py
--- a/src/energy_memory/phase2/corpus.py
+++ b/src/energy_memory/phase2/corpus.py
@@ -113,7 +113,13 @@ def load_repo_sample_splits(repo_root: Path) -> Dict[str, List[str]]:
 def load_wikitext_splits(name: str = "wikitext-2-raw-v1") -> Dict[str, List[str]]:
     if load_dataset is None:  # pragma: no cover - exercised only when dependency missing
         raise ModuleNotFoundError("datasets is required to load WikiText-2")
-    dataset = load_dataset("wikitext", name)
+    # Use the canonical Salesforce/wikitext namespace. The bare "wikitext"
+    # form worked with older HF stacks but recent huggingface_hub versions
+    # (~0.30+) ship a stricter HF URI parser that rejects any repo id
+    # without an explicit namespace, raising HfUriError. The Salesforce
+    # mirror is the current canonical home of the dataset; config names
+    # ("wikitext-2-raw-v1", "wikitext-103-raw-v1", ...) are unchanged.
+    dataset = load_dataset("Salesforce/wikitext", name)
     return {
         "train": [row["text"] for row in dataset["train"]],
         "validation": [row["text"] for row in dataset["validation"]],
'''

with open('/tmp/c3_combined.patch', 'w') as f:
    f.write(patch)
check = subprocess.run(['git', 'apply', '--check', '/tmp/c3_combined.patch'], capture_output=True, text=True)
if check.returncode == 0:
    subprocess.check_call(['git', 'apply', '/tmp/c3_combined.patch'])
    print('combined patch applied.')
else:
    if int(pre_pull or '0') >= 1 and int(pre_gram or '0') >= 1 and int(pre_wt or '0') >= 1:
        print('all three patches already in branch — skipping apply.')
    else:
        print('PATCH APPLY FAILED:'); print(check.stderr)
        raise SystemExit('Cannot continue.')

post_pull = subprocess.check_output(['grep', '-c', '--', '--lr-pull', 'experiments/c3_phase3_exit_criterion.py']).decode().strip()
post_gram = subprocess.check_output(['grep', '-c', 'kernel trick', 'src/energy_memory/phase4/consolidation.py']).decode().strip()
post_wt   = subprocess.check_output(['grep', '-c', 'Salesforce/wikitext', 'src/energy_memory/phase2/corpus.py']).decode().strip()
print(f'post-patch tracers: --lr-pull = {post_pull}; kernel-trick = {post_gram}; Salesforce/wikitext = {post_wt}')
assert int(post_pull) >= 1 and int(post_gram) >= 1 and int(post_wt) >= 1, 'patches missing'

In [ ]:
# 2. Mount Drive.
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/neuro-ai/results', exist_ok=True)
print('Drive mounted.')

In [ ]:
# 3. Install deps. WikiText needs `datasets` (pinned <3 for old plain-name compat;
#    redundant after the Salesforce/wikitext patch but kept for stability).
!pip install -q "datasets<3"
import sys, torch, numpy as np, datasets
print(f'python: {sys.version.split()[0]} | torch: {torch.__version__} | numpy: {np.__version__} | datasets: {datasets.__version__}')
print(f'cuda available: {torch.cuda.is_available()}; device 0: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

In [ ]:
# 4. Pre-warm the WikiText-2 HF cache so 30 subprocesses share the on-disk cache.
#    Parent CPU-only (does not init CUDA).
import sys; sys.path.insert(0, '/content/Neuro-AI/src')
from energy_memory.phase2.corpus import load_corpus_splits
from pathlib import Path
print('warming wikitext-2-raw-v1 cache (parent process, CPU only)...')
splits = load_corpus_splits('wikitext', Path('/content/Neuro-AI'), wikitext_name='wikitext-2-raw-v1')
print(f'  train: {len(splits["train"])} rows')
print(f'  validation: {len(splits["validation"])} rows')
print(f'  test: {len(splits["test"])} rows')
print('cache warmed.')
del splits; import gc; gc.collect()

In [ ]:
# 5. GPU info.
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

In [ ]:
# 6. SMOKE — one tiny synthetic subprocess to confirm the runtime is sane
#    BEFORE launching 40 procs.
import subprocess, sys, os
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
smoke_out = Path('reports/c3_followup_v3_smoke_2026-05-27')
smoke_out.mkdir(parents=True, exist_ok=True)
smoke_log = Path('reports/c3_followup_v3_smoke.log')
cmd = [sys.executable, 'experiments/c3_phase3_exit_criterion.py',
       '--seeds', '0', '--device', 'cuda',
       '--lr-pull', '1.0', '--lr-push', '0.05',
       '--n-consolidation-events', '100',
       '--alpha-anti', '0.01', '--repulsion-step-size', '0.05',
       '--output-dir', str(smoke_out)]
with smoke_log.open('w') as logf:
    rc = subprocess.call(cmd, stdout=logf, stderr=subprocess.STDOUT)
print(f'smoke exit code: {rc}; json: {(smoke_out / "c3_summary.json").exists()}')
print('\n=== smoke log (last 50 lines) ===')
!tail -50 {smoke_log}
if rc != 0:
    raise SystemExit('Smoke failed — abort.')
print('\nSmoke OK.')

In [ ]:
# 7. PARALLEL launch — 4 conditions × 10 seeds = 40 per-seed subprocesses.
import subprocess, os, time, signal, sys
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
PY = sys.executable

CONDITIONS = [
    # (tag, lr_pull, n_events, corpus_source)
    ('n10_synthetic_cellD', 1.0, 3000, 'synthetic'),
    ('n10_wikitext_best',   1.0, 3000, 'wikitext'),
    ('n10_wikitext_base',   0.1, 1000, 'wikitext'),
    ('n10_wikitext_mid',    0.5, 1000, 'wikitext'),
]
SEEDS = list(range(10))

# Flatten into per-seed subprocesses (40 entries).
ENTRIES = [(tag, lp, ne, corp, seed) for (tag, lp, ne, corp) in CONDITIONS for seed in SEEDS]
print(f'launching {len(ENTRIES)} per-seed subprocesses ({len(CONDITIONS)} conditions × {len(SEEDS)} seeds)')

log_root = Path('reports/c3_followup_v3_logs')
log_root.mkdir(parents=True, exist_ok=True)

def out_dir_for(tag, seed):
    return f'reports/c3_followup_v3_{tag}_seed{seed}_2026-05-27'

def launch(tag, lr_pull, n_events, corpus_source, seed):
    out_dir = out_dir_for(tag, seed)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'{tag}_seed{seed}.log'
    logf = open(log_path, 'w')
    cmd = [PY, 'experiments/c3_phase3_exit_criterion.py',
           '--seeds', str(seed), '--device', 'cuda',
           '--lr-pull', str(lr_pull), '--lr-push', '0.05',
           '--n-consolidation-events', str(n_events),
           '--alpha-anti', '0.01', '--repulsion-step-size', '0.05',
           '--corpus-source', corpus_source,
           '--output-dir', out_dir]
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
    return proc, logf, out_dir, log_path

def snapshot(remaining, total, t0):
    elapsed = (time.time() - t0) / 60
    n_remaining = len(remaining)
    n_done = total - n_remaining
    print(f'  --- snapshot at {elapsed:.1f} min — {n_done}/{total} done, {n_remaining} running ---')
    try:
        gpu = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used,utilization.gpu', '--format=csv,noheader'],
            stderr=subprocess.DEVNULL).decode().strip()
        print(f'  GPU: {gpu}')
    except Exception as e:
        print(f'  GPU snapshot failed: {e}')
    # Per-condition running counts.
    from collections import Counter
    cond_running = Counter()
    for key in remaining:
        tag = key.rsplit('_seed', 1)[0]
        cond_running[tag] += 1
    for tag, _, _, _ in CONDITIONS:
        n_run = cond_running.get(tag, 0)
        n_done_tag = len(SEEDS) - n_run
        print(f'    {tag:>22}: {n_done_tag}/{len(SEEDS)} done')

def kill_all(remaining):
    for key, (proc, logf, _, _) in remaining.items():
        try:
            proc.send_signal(signal.SIGKILL); logf.close()
        except Exception:
            pass

# Staggered launch (1.5 s × 40 = 60 s of launch).
procs = {}
for entry in ENTRIES:
    tag, lp, ne, corp, seed = entry
    key = f'{tag}_seed{seed}'
    procs[key] = launch(*entry)
    time.sleep(1.5)
print(f'all {len(procs)} cells launched  ({time.strftime("%H:%M:%S")})')

t0 = time.time()
remaining = dict(procs)
total = len(procs)
failures = []
poll_count = 0
try:
    while remaining:
        done_this_round = []
        for key, (proc, logf, out_dir, log_path) in remaining.items():
            rc = proc.poll()
            if rc is not None:
                logf.close()
                elapsed = (time.time() - t0) / 60
                json_exists = Path(out_dir, 'c3_summary.json').exists()
                ok = 'OK' if rc == 0 else f'FAILED (exit={rc})'
                print(f'  [{elapsed:5.1f} min] {key:>32}: {ok}  json={json_exists}')
                if rc != 0:
                    failures.append(key)
                    print(f'    --- last 30 lines of {log_path} ---')
                    try:
                        out = subprocess.check_output(['tail', '-30', str(log_path)],
                            stderr=subprocess.DEVNULL).decode()
                        for line in out.splitlines():
                            print(f'    | {line}')
                    except Exception as e:
                        print(f'    | (could not read log: {e})')
                    print('    --- end log ---')
                done_this_round.append(key)
        for key in done_this_round:
            del remaining[key]
        if remaining:
            poll_count += 1
            if poll_count % 3 == 0:
                snapshot(remaining, total, t0)
            time.sleep(30)
except KeyboardInterrupt:
    print('\n!!! Interrupted !!!')
    kill_all(remaining); raise

print(f'\nALL DONE in {(time.time()-t0)/60:.1f} min')
print(f'failures: {len(failures)}/{total}')
if failures:
    print('  failed keys:', failures)
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

In [ ]:
# 7b. EMERGENCY kill — Runtime → Interrupt cell 7 first, THEN run this.
import subprocess, signal, os
killed = 0
for line in subprocess.check_output(['ps', '-eo', 'pid,cmd']).decode().splitlines():
    if 'c3_phase3_exit_criterion' in line and 'grep' not in line:
        try:
            pid = int(line.split()[0])
            os.kill(pid, signal.SIGKILL); print(f'  killed {pid}'); killed += 1
        except Exception as e:
            print(f'  err: {e}')
print(f'killed {killed} workers')

In [ ]:
# 8. Copy all per-seed result dirs + logs to Drive.
import shutil, os
dst_root = '/content/drive/MyDrive/neuro-ai/results/c3_followup_v3_2026-05-27'
os.makedirs(dst_root, exist_ok=True)
CONDITIONS = ['n10_synthetic_cellD', 'n10_wikitext_best', 'n10_wikitext_base', 'n10_wikitext_mid']
SEEDS = list(range(10))
count = 0
for tag in CONDITIONS:
    for seed in SEEDS:
        src = f'reports/c3_followup_v3_{tag}_seed{seed}_2026-05-27'
        if os.path.isdir(src):
            shutil.copytree(src, f'{dst_root}/{tag}_seed{seed}', dirs_exist_ok=True)
            count += 1
if os.path.isdir('reports/c3_followup_v3_logs'):
    shutil.copytree('reports/c3_followup_v3_logs', f'{dst_root}/colab_logs', dirs_exist_ok=True)
print(f'copied {count} per-seed dirs + logs to {dst_root}')
!ls {dst_root} | head -20

In [ ]:
# 9. AGGREGATION — merge per-seed JSONs per condition and print the n=10 headline.
#    For each tag, pool per_cell_rows from all 10 seed JSONs and recompute Wilson CIs.
import json, math
from pathlib import Path

CONDITIONS = [
    # (tag, lr_pull, n_events, corpus, expected_n_seeds)
    ('n10_synthetic_cellD', 1.0, 3000, 'synthetic', 10),
    ('n10_wikitext_best',   1.0, 3000, 'wikitext',  10),
    ('n10_wikitext_base',   0.1, 1000, 'wikitext',  10),
    ('n10_wikitext_mid',    0.5, 1000, 'wikitext',  10),
]
STRATA = ('tight', 'spread', 'borderline')

# Local Wilson interval — avoids needing to import from the repo's helpers.
def wilson(successes, trials, z=1.96):
    if trials == 0:
        return (0.0, 0.0, 0.0)
    p = successes / trials
    n = trials
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = (z * math.sqrt(p*(1-p)/n + z*z/(4*n*n))) / denom
    return (p, max(0.0, center - half), min(1.0, center + half))

def merge_tag(tag):
    """Return (merged_aggregated_dict, n_seeds_present, modes_seen)."""
    rows = []
    seeds_present = set()
    modes_seen = set()
    for seed in range(10):
        p = Path(f'reports/c3_followup_v3_{tag}_seed{seed}_2026-05-27/c3_summary.json')
        if not p.exists():
            continue
        d = json.loads(p.read_text())
        seeds_present.add(seed)
        for r in d['per_cell_rows']:
            rows.append(r)
            modes_seen.add(r['theta_prime_mode'])
    aggregated = {}
    for mode in sorted(modes_seen):
        aggregated[mode] = {}
        for is_control in (False, True):
            key = 'shuffled_control' if is_control else 'standard'
            aggregated[mode][key] = {}
            for stratum in STRATA:
                tot_s = 0; tot_t = 0
                for r in rows:
                    if r['theta_prime_mode'] != mode or r['is_control'] != is_control:
                        continue
                    cell = r['per_stratum'][stratum]
                    tot_s += int(cell['successes']); tot_t += int(cell['trials'])
                mean_v, lo, hi = wilson(tot_s, tot_t)
                aggregated[mode][key][stratum] = {
                    'successes': tot_s, 'trials': tot_t,
                    'recall_at_k': mean_v, 'wilson_lower': lo, 'wilson_upper': hi,
                }
        aggregated[mode]['delta_standard_minus_control'] = {}
        for stratum in STRATA:
            s = aggregated[mode]['standard'][stratum]
            c = aggregated[mode]['shuffled_control'][stratum]
            aggregated[mode]['delta_standard_minus_control'][stratum] = {
                'delta_recall_at_k': s['recall_at_k'] - c['recall_at_k'],
                'standard_trials': s['trials'],
                'control_trials': c['trials'],
                'ci_disjoint_standard_beats_control': s['wilson_lower'] > c['wilson_upper'],
            }
    return aggregated, sorted(seeds_present), sorted(modes_seen)

# Write merged summaries to Drive too.
import os
merged_root = '/content/drive/MyDrive/neuro-ai/results/c3_followup_v3_2026-05-27/_merged_n10'
os.makedirs(merged_root, exist_ok=True)

print(f'{"cell":>22} {"lrP":>5} {"n_ev":>5} {"corpus":>10} {"seeds":>5} {"mode":>11} {"stratum":>11}  '
      f'{"std R@K":>22}  {"ctrl R@K":>22}  {"Δ":>7}  disjoint?  n_std  n_ctrl')
any_disjoint = []
for tag, lr_pull, n_events, corpus, exp_seeds in CONDITIONS:
    agg, seeds_present, modes = merge_tag(tag)
    n_present = len(seeds_present)
    if n_present == 0:
        print(f'{tag:>22} {lr_pull:>5} {n_events:>5} {corpus:>10}  ALL MISSING')
        continue
    if n_present < exp_seeds:
        print(f'  (warning: {tag} has only {n_present}/{exp_seeds} seeds present: {seeds_present})')
    # Persist merged JSON.
    out = {'tag': tag, 'lr_pull': lr_pull, 'n_events': n_events, 'corpus': corpus,
           'seeds_present': seeds_present, 'aggregated': agg}
    with open(f'{merged_root}/{tag}.json', 'w') as f:
        json.dump(out, f, indent=2)
    for mode in modes:
        for stratum in STRATA:
            s = agg[mode]['standard'][stratum]
            c = agg[mode]['shuffled_control'][stratum]
            dl = agg[mode]['delta_standard_minus_control'][stratum]
            if s['trials'] == 0 and c['trials'] == 0:
                continue
            std_str  = f'{s["recall_at_k"]:.3f} [{s["wilson_lower"]:.3f},{s["wilson_upper"]:.3f}]'
            ctrl_str = f'{c["recall_at_k"]:.3f} [{c["wilson_lower"]:.3f},{c["wilson_upper"]:.3f}]'
            disj = 'YES' if dl['ci_disjoint_standard_beats_control'] else 'no'
            print(f'{tag:>22} {lr_pull:>5} {n_events:>5} {corpus:>10} {n_present:>5} {mode:>11} {stratum:>11}  '
                  f'{std_str:>22}  {ctrl_str:>22}  {dl["delta_recall_at_k"]:>+7.3f}  {disj:>9}  '
                  f'{s["trials"]:>5}  {c["trials"]:>5}')
            if dl['ci_disjoint_standard_beats_control'] and s['trials'] > 0 and c['trials'] > 0:
                any_disjoint.append((tag, mode, stratum, dl['delta_recall_at_k']))

print()
if any_disjoint:
    print('=== POSITIVE SIGNAL — CI-disjoint strata ===')
    for tag, mode, stratum, delta in any_disjoint:
        print(f'  {tag} | {mode}/{stratum} | Δ={delta:+.3f}')
    print()
    print('Next step: if n10_wikitext_best is among these, that is a Phase 3 GRADUATION CANDIDATE.')
    print('Write a numbered report, update STATUS.md, and reopen Phase 5′ planning.')
else:
    print('=== ALL CI-OVERLAPPING — null at n=10 ===')
    print('Consolidation-strength × corpus-thinness joint test does not break the Path α null.')
    print('Recommended next path: Path γ — Phase 3 mechanism redesign per literature-and-principles.md.')